In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

In [4]:
# подгружаем .env
load_dotenv()

True

In [5]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [6]:
# Создадим соединения
src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [7]:
# Пример выгрузки данных из БД
TABLE = ''
SQL = f'SELECT * from buildings_flats_joined'
data = pd.read_sql(SQL, dst_conn)

In [13]:
# Собранный датасет из двух таблиц
data

,id,building_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,flat_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price
0,4596,2708,1959,1,55.752495,37.678139,3.00,47,5,false,4595,1,8.0,53.000000,4,false,false,71.400002,15500000.0
1,4597,6454,1966,1,55.879047,37.623222,2.64,108,9,true,4596,1,5.5,18.000000,1,false,false,29.700001,7390000.0
2,4598,10331,1973,4,55.589203,37.647827,2.70,191,12,true,4597,9,6.6,45.099998,3,false,false,62.900002,13900000.0
3,4599,22421,2013,2,55.867252,37.686005,3.00,152,22,true,4598,4,20.1,65.000000,3,false,false,111.699997,23000000.0
4,4600,22421,2013,2,55.867252,37.686005,3.00,152,22,true,4599,6,11.0,35.700001,2,false,false,56.700001,13500000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,36996,18633,2002,4,55.852596,37.640335,2.74,332,17,true,36995,2,11.0,0.000000,3,false,false,74.000000,15000000.0
141358,36997,13988,1983,4,55.884796,37.490021,2.64,255,16,true,36996,6,8.4,18.900000,1,false,false,36.099998,6900000.0
141359,36998,11566,1976,4,55.580997,37.669308,2.48,428,12,true,36997,8,7.0,0.000000,2,false,false,48.000000,9000000.0
141360,36999,14663,1986,6,55.609524,37.730076,2.48,109,16,true,36998,2,9.8,0.000000,1,false,false,36.900002,7500000.0


In [14]:
# Выделяем категориальные и числовые переменные
num_features=data.select_dtypes(include=['float','int'])
cat_features=data.select_dtypes(include=['object'])

In [15]:
# Поиск дубликатов
data.duplicated(subset=['building_id', 'flat_id'], keep=False).sum()

np.int64(0)

In [16]:
# Поиск пропусков
data.isnull().sum()

id                   0
building_id          0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
flat_id              0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
dtype: int64

In [17]:
# Поиск выбросов
num_cols = num_features.select_dtypes(['float']).columns
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in num_cols:
    Q1 = num_features[col].quantile(0.25)# Ваш код здесь #
    Q3 = num_features[col].quantile(0.75) # Ваш код здесь #
    IQR = Q3-Q1 # Ваш код здесь #
    margin = 1.5*IQR # Ваш код здесь #
    lower = Q1 - margin  # Ваш Код здесь #
    upper = Q3 + margin# Ваш Код здесь #
    potential_outliers[col] = ~num_features[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)

print(data[outliers])

           id  building_id  build_year  building_type_int   latitude  \
3        4599        22421        2013                  2  55.867252   
5        4601        22421        2013                  2  55.867252   
34       4630         2338        1958                  1  55.789494   
57       4653        16664        1996                  4  55.868111   
66       4662         5717        1964                  6  55.794125   
...       ...          ...         ...                ...        ...   
141306  36945        16137        1994                  4  55.971245   
141327  36966        21165        2009                  2  55.603111   
141331  36970         1523        1955                  1  55.781372   
141342  36981        23366        2016                  2  55.675621   
141346  36985        20411        2007                  2  55.647194   

        longitude  ceiling_height  flats_count  floors_total has_elevator  \
3       37.686005            3.00          152            